In [1]:
import sys, json, re, time
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession, SDCFaultInjector, finite_key_output_length
from qne.cascade.key import key_from_sifted_json
from qne.cascade.sweep_utils import run_condition_sweep, summarize_outcomes

SLICE_NAME = 'qfabric-bb84-2'
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

import json as _json
GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

if not (GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists()):
    raise FileNotFoundError(
        "PE-split generation-only key files not found. Run "
        "10_sdc_real.ipynb's key-loading cell first."
    )

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = _json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]
print(f"Loaded {alice_key.get_nr_bits()} generation bits, k={k_pe}, real_qber = {real_qber:.4f}")

# Re-sync remote nodes to this exact (PE-split, generation-only) key pair
alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Setup complete, nodes synced")


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
Loaded 3488 generation bits, k=388, real_qber = 0.0077
Setup complete, nodes synced


In [ ]:
"""
Scaled-up Mock dose-response: Toeplitz, final-key, reconciliation, and
verify-digest, all at n=50 per probability point.
"""
from randextract import ToeplitzHashing
import numpy as np

probs_th = [0.5, 0.3, 0.1, 0.05, 0.03, 0.01, 0.003, 0.001]
probs_recon = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
all_dfs, summary_rows = {}, []

for fault_type, probs, kwarg_name in [("toeplitz", probs_th, "toeplitz_prob"),
                                        ("final_key", probs_th, "final_key_prob"),
                                        ("reconciliation", probs_recon, "reconciliation_prob")]:
    for prob in probs:
        label = f"{fault_type}_{prob}"
        print(f"Running {label} (n=50)...")
        df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=50, **{kwarg_name: prob})
        df_cond["fault_type"] = fault_type
        df_cond["prob"] = prob
        all_dfs[label] = df_cond
        summary_rows.append(summarize_outcomes(df_cond, label))

# --- Verify-digest sweep: split uses the PLACEHOLDER length (not finite-key) ---
n_key0 = alice_key.get_nr_bits()
ell = ToeplitzHashing.calculate_length(
    extractor_type="quantum", input_length=n_key0,
    relative_source_entropy=0.5, error_bound=1e-6,
)
t_placeholder = max(1, int(np.ceil(-np.log2(1e-10))))  # same fallback t the driver uses in placeholder mode
digest_length = max(1, int(np.ceil(t_placeholder)))
t_verify = max(2 * digest_length, 32)
print(f"\nVerify-digest split (placeholder length): ell={ell}, t_verify={t_verify}, digest_length={digest_length}")

for prob in probs_th:
    label = f"verify_digest_{prob}"
    print(f"Running {label} (n=50)...")
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=50,
                                     verify_digest_prob=prob, ell=ell, t_verify=t_verify,
                                     digest_length=digest_length)
    df_cond["fault_type"] = "verify_digest"
    df_cond["prob"] = prob
    all_dfs[label] = df_cond
    summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_mock_doseresponse_n50.csv"), index=False)
print(f"Saved {len(full_df)} rows -> sdc_mock_doseresponse_n50.csv")

In [ ]:
"""
Cell A: Toeplitz-matrix faults, real channel, n=20 per probability,
all three length modes (placeholder, asymptotic, finite_key) for side-by-side comparison.
"""
from qne.cascade.sweep_utils import run_real_channel_trial

length_modes = ["placeholder", "asymptotic"]
rows = []

for length_mode in length_modes:
    for prob in probs_th:
        print(f"\n=== toeplitz prob={prob}, length_mode={length_mode} (n=20) ===")
        for run in range(20):
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, toeplitz_prob=prob)
            result["fault_type"] = "toeplitz"
            result["prob"] = prob
            result["length_mode"] = length_mode
            rows.append(result)
            if run % 5 == 0:
                print(f"  run {run}: keys_match={result.get('keys_match')}, "
                      f"secure_key_length={result.get('secure_key_length')}")

df_toeplitz_n20 = pd.DataFrame(rows)
df_toeplitz_n20.to_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_n20_all_modes.csv"), index=False)
print(f"\nSaved {len(df_toeplitz_n20)} rows -> sdc_realchannel_toeplitz_n20_all_modes.csv")

In [ ]:
"""
Final-key faults, real channel, n=20 per probability, all three length modes.
Saves incrementally -- each row appended immediately, so a disconnect never
loses more than one trial.
"""
import os

final_key_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_n20_all_modes.csv")

if os.path.exists(final_key_output_path):
    df_existing = pd.read_csv(final_key_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(final_key_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_th:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, final_key_prob=prob)
            result["fault_type"] = "final_key"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(final_key_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

print("\nFinal-key sweep complete.")
print(pd.read_csv(final_key_output_path).groupby(["length_mode", "prob"]).size())

In [ ]:
from qne.cascade.sweep_utils import run_real_channel_trial, run_real_channel_reconciliation_trial
"""
Reconciliation-state faults, real channel, n=20 per probability, all three length modes.
Same incremental-save pattern as the final-key sweep above.
"""
recon_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_n20_all_modes.csv")

if os.path.exists(recon_output_path):
    df_existing = pd.read_csv(recon_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(recon_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_recon:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, real_qber, run, seed,
                                                               reconciliation_prob=prob, k=k_pe,
                                                               length_mode=length_mode)
            result["fault_type"] = "reconciliation"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(recon_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"non_convergent={result.get('non_convergent')} [saved]")

print("\nReconciliation sweep complete.")
print(pd.read_csv(recon_output_path).groupby(["length_mode", "prob"]).size())

In [ ]:
"""
Verification-digest faults, real channel, n=20 per probability, all three length modes.
Same incremental-save pattern as the final-key/reconciliation sweeps above.
"""
import os

verify_digest_output_path = str(PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_n20_all_modes.csv")

if os.path.exists(verify_digest_output_path):
    df_existing = pd.read_csv(verify_digest_output_path)
    completed = set(zip(df_existing["length_mode"], df_existing["prob"], df_existing["run"]))
    print(f"Resuming: {len(completed)} trials already done")
else:
    completed = set()
    print("Starting fresh")

header_written = os.path.exists(verify_digest_output_path)

for length_mode in ["placeholder", "asymptotic"]:
    for prob in probs_th:
        for run in range(20):
            if (length_mode, prob, run) in completed:
                continue

            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                               length_mode=length_mode, verify_digest_prob=prob)
            result["fault_type"] = "verify_digest"
            result["prob"] = prob
            result["length_mode"] = length_mode

            row_df = pd.DataFrame([result])
            row_df.to_csv(verify_digest_output_path, mode="a", header=not header_written, index=False)
            header_written = True

            print(f"  mode={length_mode}, prob={prob}, run={run}: "
                  f"keys_match={result.get('keys_match')}, "
                  f"verification_passed={result.get('verification_passed')} [saved]")

print("\nVerification-digest sweep complete.")
print(pd.read_csv(verify_digest_output_path).groupby(["length_mode", "prob"]).size())

In [ ]:
# Quick completeness check across all mock + real-channel files
# (length-mode comparison + verification coverage)
files_to_check = [
    "sdc_mock_doseresponse_n50.csv",
    "sdc_realchannel_toeplitz_n20_all_modes.csv",
    "sdc_realchannel_finalkey_n20_all_modes.csv",
    "sdc_realchannel_reconciliation_n20_all_modes.csv",
    "sdc_realchannel_verifydigest_n20_all_modes.csv",
]

for fname in files_to_check:
    path = PROJECT_DIR / "results" / fname
    if not path.exists():
        print(f"{fname}: NOT FOUND\n")
        continue

    df = pd.read_csv(str(path))
    print(f"=== {fname} ({len(df)} rows) ===")

    if "length_mode" in df.columns:
        print(df.groupby(["length_mode", "prob"]).size().unstack(fill_value=0))
    else:
        print(df.groupby(["fault_type", "prob"]).size().unstack(fill_value=0)
              if "fault_type" in df.columns else df.groupby("prob").size())

    if "verification_passed" in df.columns:
        n_with_verification = df["verification_passed"].notna().sum()
        print(f"  Rows with verification data: {n_with_verification}/{len(df)}")
        if n_with_verification == 0:
            print("  NOTE: no verification data in this file -- likely run with "
                  "ell=None (no split performed). Do not treat NaN here as "
                  "'verification passed' or 'verification failed'.")
        elif n_with_verification < len(df):
            missing_types = df[df["verification_passed"].isna()]["fault_type"].unique() \
                if "fault_type" in df.columns else "unknown"
            print(f"  Partial coverage -- fault types missing verification data: {missing_types}")
    print()

In [ ]:
"""
Real Channel (n=20/point/mode): length-mode comparison across all four
fault types, PLUS an explicit undetected-silent-corruption panel now that
verification_passed is available in the real-channel data.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def wilson_ci(successes, n, confidence=0.95):
    if n == 0:
        return 0.0, 0.0, 0.0
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    p_hat = successes / n
    denom = 1 + z**2 / n
    center = (p_hat + z**2 / (2 * n)) / denom
    half_width = (z / denom) * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    return p_hat, max(0, center - half_width), min(1, center + half_width)


def rate_by_prob(df, prob_col, success_col):
    probs = sorted(df[prob_col].unique())
    rates, lo, hi = [], [], []
    for p in probs:
        sub = df[df[prob_col] == p]
        n = len(sub)
        successes = sub[success_col].fillna(False).astype(bool).sum()
        rate, rlo, rhi = wilson_ci(successes, n)
        rates.append(rate); lo.append(rlo); hi.append(rhi)
    return probs, rates, lo, hi


mode_colors = {"placeholder": "tab:blue", "asymptotic": "tab:orange"}

df_toeplitz = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_toeplitz_n20_all_modes.csv"))
df_finalkey = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_finalkey_n20_all_modes.csv"))
df_recon = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_realchannel_reconciliation_n20_all_modes.csv"))

df_toeplitz["mismatch"] = ~df_toeplitz["keys_match"].fillna(False)
df_finalkey["mismatch"] = ~df_finalkey["keys_match"].fillna(False)

# Load verify_digest data only if it exists -- optional given FABRIC cost
verifydigest_path = PROJECT_DIR / "results" / "sdc_realchannel_verifydigest_n20_all_modes.csv"
df_verifydigest = pd.read_csv(str(verifydigest_path)) if verifydigest_path.exists() else None

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- (0,0) Toeplitz mismatch rate by length mode ---
for mode in ["placeholder", "asymptotic"]:
    sub = df_toeplitz[df_toeplitz["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[0, 0].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[0, 0].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[0, 0].set_xscale('log'); axes[0, 0].set_xlabel('Fault probability')
axes[0, 0].set_ylabel('Mismatch rate')
axes[0, 0].set_title('Toeplitz-matrix fault')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# --- (0,1) Final-key mismatch rate by length mode ---
for mode in ["placeholder", "asymptotic"]:
    sub = df_finalkey[df_finalkey["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "prob", "mismatch")
    axes[0, 1].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[0, 1].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[0, 1].set_xscale('log'); axes[0, 1].set_xlabel('Fault probability')
axes[0, 1].set_title('Final-key fault')
axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

# --- (1,0) Reconciliation non-convergence rate by length mode ---
for mode in ["placeholder", "asymptotic"]:
    sub = df_recon[df_recon["length_mode"] == mode]
    probs, rates, lo, hi = rate_by_prob(sub, "reconciliation_prob", "non_convergent")
    axes[1, 0].plot(probs, rates, marker='o', label=mode, color=mode_colors[mode])
    axes[1, 0].fill_between(probs, lo, hi, alpha=0.15, color=mode_colors[mode])
axes[1, 0].set_xscale('log'); axes[1, 0].set_xlabel('Reconciliation fault probability')
axes[1, 0].set_ylabel('Non-convergence rate')
axes[1, 0].set_title('Reconciliation-state fault')
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# --- (1,1) NEW: undetected-silent-corruption rate, aggregated across all
# probabilities, by length mode -- the paper's core headline metric,
# now measurable directly from the real-channel data since
# verification_passed is populated for toeplitz/final_key here. ---
undetected_rows = []
for label, df_src in [("toeplitz", df_toeplitz), ("final_key", df_finalkey)]:
    for mode in ["placeholder", "asymptotic"]:
        sub = df_src[df_src["length_mode"] == mode]
        confirmed_mismatch = sub[sub["mismatch"] == True]
        n_mismatch = len(confirmed_mismatch)
        n_undetected = (confirmed_mismatch["verification_passed"] == True).sum()
        rate = n_undetected / n_mismatch if n_mismatch > 0 else np.nan
        undetected_rows.append({"fault_type": label, "length_mode": mode,
                                  "n_mismatch": n_mismatch, "n_undetected": n_undetected,
                                  "undetected_rate": rate})

df_undetected = pd.DataFrame(undetected_rows)
print("=== Undetected silent-corruption rate (confirmed mismatches only) ===")
print(df_undetected.to_string(index=False))

pivot = df_undetected.pivot(index="fault_type", columns="length_mode", values="undetected_rate")
pivot = pivot[["placeholder", "asymptotic"]]  # consistent column order
pivot.plot(kind="bar", ax=axes[1, 1], color=[mode_colors[m] for m in pivot.columns])
axes[1, 1].set_ylabel('Undetected rate (of confirmed mismatches)')
axes[1, 1].set_title('Undetected silent corruption, by length mode')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)
axes[1, 1].legend(title="length mode")
axes[1, 1].grid(alpha=0.3, axis='y')

plt.suptitle('Real Channel (n=20/point/mode): Fault Outcomes and Detection Across Length Modes')
plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "results" / "fig_c_lengthmode_comparison_with_verification.png"), dpi=150)
plt.show()

# --- Separate: verify_digest false-abort rate by length mode and probability ---
if df_verifydigest is not None:
    print("\n=== verify_digest: false-abort rate (verification_passed=False despite keys_match=True) ===")
    df_verifydigest["false_abort"] = (df_verifydigest["keys_match"] == True) & \
                                       (df_verifydigest["verification_passed"] == False)
    fa_pivot = df_verifydigest.groupby(["length_mode", "prob"])["false_abort"].mean().unstack(level=0)
    print(fa_pivot.to_string())
else:
    print("\nsdc_realchannel_verifydigest_n20_all_modes.csv not found -- "
          "skipping verify_digest false-abort analysis (optional sweep, not yet run).")